## 환경 확인

upper 세트로 생성해 실사용 구도의 실패를 재현한다. 결과물은 레버 A(복원 후처리)의
입력이 된다.

측정 결과 upper 세트의 latent 눈 폭은 2.3~5.3 으로, 기존 normal 세트(4.4~10.5)의
아래쪽에 몰려 있다. 6장 중 4장은 MIN_FACE_WIDTH 검증에 걸리므로 validate 를
거치지 않고 파이프라인을 직접 호출한다.

6장을 조합 3·5 양쪽으로 돌려 12장을 만든다. 모델 전환 비용이 생성보다 크므로
조합별로 묶어 돌린다.

In [ ]:
import subprocess

import torch

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout
)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

from google.colab import drive

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/saloncut_data"
!ls {BASE}
!echo "--- upper ---" && ls {BASE}/test_images/upper 2>/dev/null || echo "upper 없음"
!echo "--- ref_faces ---" && ls {BASE}/ref_faces 2>/dev/null | head -40

## 레포·모델 준비

레포의 `image_gen` 모듈을 그대로 쓴다. 실서버와 같은 코드로 돌려야 결과가
실사용 상황을 반영한다. validate 만 건너뛴다.

참조 얼굴은 두 개로 고정한다.
  여성 5장 — ref-01 (한국 20 여 매력)
  남성 1장 — ref-05 (한국 20 남 매력)

나이·인상까지 흩뜨리면 실패 원인이 섞이므로, 이번 실험의 변수는 얼굴 크기 하나다.

In [ ]:
!git clone -q https://github.com/qja0707/SalonCutAI.git /content/SalonCutAI
%cd /content/SalonCutAI/backend
!git checkout -q dev
!pip install -q diffusers transformers accelerate insightface onnxruntime-gpu mediapipe opencv-python

import os
import sys

os.environ["SALON_STORAGE_DIR"] = "/content/storage"
os.environ["IMAGE_GEN_ENABLED"] = "1"
sys.path.insert(0, "/content/SalonCutAI/backend")

from src.ai_engine.image_gen import downloads

downloads.ensure_models()
print("모델 준비 완료")

## 조합 3 — 참조 얼굴 모드

6장을 연속으로 돌린다. 모델 전환 비용이 생성보다 크므로 조합별로 묶는다.

validate 를 거치지 않는다. 6장 중 4장이 MIN_FACE_WIDTH 에 걸리는데, 반려되는
사진이야말로 실사용 실패의 핵심이라 반드시 포함해야 한다.

seed 42 고정. 사진마다 결과가 다른 이유를 얼굴 크기로만 돌리기 위함이다.

In [ ]:
import time
from pathlib import Path

from PIL import Image

from src.ai_engine.image_gen import combo3, loader, settings

settings.IMAGE_GEN_ENABLED = True

UPPER = Path("/content/drive/MyDrive/saloncut_data/test_images/upper")
REF = Path("/content/drive/MyDrive/saloncut_data/ref_faces")
OUT = Path("/content/out")
OUT.mkdir(exist_ok=True)

SEED = 42
REF_MAP = {"upper_06_asian_male": "ref-05"}  # 나머지는 ref-01

files = sorted(UPPER.glob("*.jpg"))
print(f"{len(files)}장")

for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    t0 = time.time()

    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    out.save(OUT / f"c3_{path.stem}.png")
    print(f"{path.stem:<26} ref={ref_id}  {time.time() - t0:5.1f}초  {out.size}")

print("\n조합 3 완료")

## 조합 3 결과 확인

후처리 전 생성 원본이다. 눈 영역을 확대해 실패가 재현되는지 본다.

latent 눈 폭 순으로 배열한다. 값이 낮을수록 생성 시점 눈이 작다.

In [ ]:
import matplotlib.pyplot as plt

# 측정값. latent 눈 폭 오름차순
LATENT = {
    "upper_06_asian_male": 2.3,
    "upper_05_asian_glasses": 2.5,
    "upper_04_asian_landscape": 3.2,
    "upper_02_west_small_face": 3.3,
    "upper_03_asian_tied": 4.7,
    "upper_01_asian_long_dark": 5.3,
}

order = sorted(LATENT, key=LATENT.get)

fig, axes = plt.subplots(2, 6, figsize=(22, 12))
for i, name in enumerate(order):
    axes[0, i].imshow(Image.open(UPPER / f"{name}.jpg"))
    axes[0, i].set_title(f"원본  {name[:14]}", fontsize=9)
    axes[1, i].imshow(Image.open(OUT / f"c3_{name}.png"))
    axes[1, i].set_title(f"조합3  latent {LATENT[name]}", fontsize=9)
    for r in (0, 1):
        axes[r, i].axis("off")
plt.tight_layout()
plt.show()

## 후처리 적용

pipeline._run_reference_mode() 의 후처리 3단계를 그대로 적용한다.
여기까지가 실서버 최종 얼굴이다. 이후 비율 크롭은 화면 크기만 바꾸므로
얼굴 픽셀은 달라지지 않는다.

  1. 색 정합 alpha 1.0
  2. 어파인 정렬 + 헤어 복원 재합성
  3. 고주파 이식 0.5

생성 원본과 최종본을 모두 남겨 레버 F(후처리 재검토) 판단에 쓴다.

In [ ]:
from src.ai_engine.image_gen import compose, masks

FINAL = Path("/content/final")
FINAL.mkdir(exist_ok=True)

for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    t0 = time.time()

    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    face_mask = masks.build_face_mask(img_r)
    if face_mask is None:
        print(f"{path.stem:<26} 얼굴 마스크 실패")
        continue
    hair_mask = masks.build_hair_mask(img_r)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    out = compose.color_transfer(out, img_r, gen_mask)
    comp = compose.align_then_recompose(img_r, out, face_mask, hair_mask)
    final = compose.transfer_high_freq(comp, img_r, gen_mask)

    final.save(FINAL / f"c3_{path.stem}.png")
    print(f"{path.stem:<26} {time.time() - t0:5.1f}초  {final.size}")

print("\n후처리 완료")

## 3단 비교

원본 · 후처리 전 · 후처리 후를 나란히 본다. latent 눈 폭 오름차순이다.

후처리 전 헤어가 바뀌어 보였던 것이 재합성으로 복구되는지 확인한다.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(22, 17))
rows = [("원본", UPPER, "{}.jpg"), ("생성", OUT, "c3_{}.png"), ("최종", FINAL, "c3_{}.png")]

for i, name in enumerate(order):
    for r, (label, d, pat) in enumerate(rows):
        axes[r, i].imshow(Image.open(d / pat.format(name)))
        axes[r, i].set_title(f"{label}  {name[6:16]}  L{LATENT[name]}", fontsize=9)
        axes[r, i].axis("off")
plt.tight_layout()
plt.show()

## 그림 저장

보고서 수록을 위해 report_figures/fig{번호}_{내용}.png 로 저장한다.
기존 산출물이 fig81 까지 있으므로 82 부터 이어간다.

In [ ]:
FIG = Path("/content/drive/MyDrive/saloncut_data/outputs/report_figures")


def savefig(num, name):
    """현재 figure 를 report_figures 에 저장한다."""
    path = FIG / f"fig{num}_{name}.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    print(f"저장  {path.name}")


# --- fig82 상반신 세트 원본 ---

fig, axes = plt.subplots(1, 6, figsize=(22, 6))
for ax, name in zip(axes, order):
    ax.imshow(Image.open(UPPER / f"{name}.jpg"))
    ax.set_title(f"{name[6:]}  L{LATENT[name]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
savefig(82, "upper_set_original")
plt.show()

# --- fig83 조합 3 생성 원본 ---

fig, axes = plt.subplots(1, 6, figsize=(22, 6))
for ax, name in zip(axes, order):
    ax.imshow(Image.open(OUT / f"c3_{name}.png"))
    ax.set_title(f"{name[6:]}  L{LATENT[name]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
savefig(83, "combo3_before_compose")
plt.show()

# --- fig84 3단 비교 ---

fig, axes = plt.subplots(3, 6, figsize=(22, 17))
rows = [("원본", UPPER, "{}.jpg"), ("생성", OUT, "c3_{}.png"), ("최종", FINAL, "c3_{}.png")]
for i, name in enumerate(order):
    for r, (label, d, pat) in enumerate(rows):
        axes[r, i].imshow(Image.open(d / pat.format(name)))
        axes[r, i].set_title(f"{label}  {name[6:16]}  L{LATENT[name]}", fontsize=9)
        axes[r, i].axis("off")
plt.tight_layout()
savefig(84, "combo3_3stage_compare")
plt.show()

In [ ]:
import shutil

DRIVE_OUT = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3")
(DRIVE_OUT / "raw").mkdir(parents=True, exist_ok=True)
(DRIVE_OUT / "final").mkdir(parents=True, exist_ok=True)

for src, sub in ((OUT, "raw"), (FINAL, "final")):
    for p in sorted(src.glob("*.png")):
        shutil.copy(p, DRIVE_OUT / sub / p.name)

print(f"저장 위치  {DRIVE_OUT}")
!ls -R {DRIVE_OUT}

## 후처리 단계 분해

upper_02 가 최종본에서 원본 얼굴로 상당히 되돌아갔다. 후처리 3단계 중 어디서
일어나는지 단계별로 나눠 본다.

  A 생성 원본
  B A + 색 정합
  C B + 어파인 정렬·헤어 복원 재합성
  D C + 고주파 이식  ← 현재 최종

각 단계에서 원본 얼굴과의 InsightFace 코사인을 잰다. 값이 오르면 원본 픽셀이
돌아온 것이다. 8/12 실험에서 고주파 강도 1.0 일 때 0.1767 → 0.2394 로 오른
전례가 있다.

In [ ]:
!apt-get install -qq fonts-nanum > /dev/null
!fc-cache -fv > /dev/null

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
import numpy as np

from src.ai_engine.image_gen import loader as gen_loader

STAGES = ("A_생성", "B_색정합", "C_재합성", "D_고주파")

STAGE_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/stages")
STAGE_DIR.mkdir(parents=True, exist_ok=True)


def identity(img_a, img_b):
    """두 이미지 얼굴의 코사인. 검출 실패면 None."""
    app = gen_loader.get_face_app()
    embs = []
    for im in (img_a, img_b):
        f = app.get(np.array(im)[:, :, ::-1])
        if not f:
            return None
        embs.append(f[0].normed_embedding)
    return float(np.dot(embs[0], embs[1]))


rows = []
stage_imgs = {}

for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    a = out
    b = compose.color_transfer(a, img_r, gen_mask)
    c = compose.align_then_recompose(img_r, b, face_mask, hair_mask)
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    for tag, im in zip(STAGES, (a, b, c, d)):
        im.save(STAGE_DIR / f"{path.stem}_{tag}.png")
    img_r.save(STAGE_DIR / f"{path.stem}_원본1024.png")

    stage_imgs[path.stem] = (img_r, [a, b, c, d])
    rows.append(
        (path.stem, [identity(img_r, x) for x in (a, b, c, d)])
    )
    print(f"{path.stem} 완료")

# --- 출력 ---

print(f"\n{'file':<26}" + "".join(f"{s:>12}" for s in STAGES))
for name, vals in rows:
    print(
        f"{name:<26}"
        + "".join(f"{'-' if v is None else f'{v:.4f}':>12}" for v in vals)
    )

## 마스크 커버리지

재합성이 마스크 밖을 원본으로 덮는다. 코사인이 이 단계에서 +0.13~0.37 뛰었으므로
마스크가 실제 얼굴보다 작을 가능성이 있다.

InsightFace 얼굴 박스를 기준으로 gen_mask 가 그 안을 얼마나 덮는지 잰다.
  커버리지 = (마스크 ∩ 얼굴박스) 면적 / 얼굴박스 면적

gen_mask 는 face_mask 에서 hair_mask 를 뺀 것이라 앞머리가 내려온 사진은
낮게 나오는 것이 정상이다. 그 경우와 마스크 자체가 작은 경우를 구분해야 한다.

In [ ]:
STAGE_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/stages")

rows_mask = []
for path in files:
    img_r = Image.open(STAGE_DIR / f"{path.stem}_원본1024.png").convert("RGB")

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    fm_a = np.array(face_mask) > 127
    hm_a = np.array(hair_mask) > 127
    gm_a = np.array(gen_mask) > 127

    # InsightFace 얼굴 박스
    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    if not det:
        rows_mask.append((path.stem, None, None, None, None))
        continue
    x1, y1, x2, y2 = [int(v) for v in det[0].bbox]
    box_area = (x2 - x1) * (y2 - y1)

    rows_mask.append(
        (
            path.stem,
            gm_a[y1:y2, x1:x2].sum() / box_area,   # gen 커버리지
            fm_a[y1:y2, x1:x2].sum() / box_area,   # face 커버리지
            hm_a[y1:y2, x1:x2].sum() / box_area,   # 헤어 침범
            box_area,
        )
    )

print(f"{'file':<26}{'gen커버':>10}{'face커버':>10}{'헤어침범':>10}{'박스면적':>10}")
for name, g, f, h, a in rows_mask:
    if g is None:
        print(f"{name:<26}{'검출실패':>10}")
        continue
    print(f"{name:<26}{g:>10.3f}{f:>10.3f}{h:>10.3f}{a:>10}")

## 헤어 팽창 기여분

HAIR_DILATE_PX = 20 은 앞머리 SSIM 기준으로 얼빡 세트에서 정한 절대값이다.
얼굴 박스가 작은 사진에서는 상대적으로 크게 작용한다.

팽창 없는 헤어 마스크와 나란히 재서 침범 중 얼마가 팽창 때문인지 가른다.
원본에 앞머리가 실제로 내려온 것과 구분해야 한다.

In [ ]:
rows_d = []
overlays = {}

for path in files:
    img_r = Image.open(STAGE_DIR / f"{path.stem}_원본1024.png").convert("RGB")

    face_mask = masks.build_face_mask(img_r)
    hair_0 = masks.build_hair_mask(img_r, dilate=0)
    hair_20 = masks.build_hair_mask(img_r)
    gen_0 = masks.build_gen_mask(face_mask, hair_0)
    gen_20 = masks.build_gen_mask(face_mask, hair_20)

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    x1, y1, x2, y2 = [int(v) for v in det[0].bbox]
    area = (x2 - x1) * (y2 - y1)

    def cov(m):
        return (np.array(m) > 127)[y1:y2, x1:x2].sum() / area

    rows_d.append(
        (path.stem, cov(hair_0), cov(hair_20), cov(gen_0), cov(gen_20), x2 - x1)
    )
    overlays[path.stem] = (img_r, face_mask, hair_20, gen_20, (x1, y1, x2, y2))

print(
    f"{'file':<26}{'헤어0':>9}{'헤어20':>9}{'팽창분':>9}"
    f"{'gen0':>9}{'gen20':>9}{'손실':>9}{'얼굴폭':>8}"
)
for name, h0, h20, g0, g20, w in rows_d:
    print(
        f"{name:<26}{h0:>9.3f}{h20:>9.3f}{h20 - h0:>9.3f}"
        f"{g0:>9.3f}{g20:>9.3f}{g0 - g20:>9.3f}{w:>8}"
    )

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 7))
for ax, name in zip(axes, order):
    img_r, fm, hm, gm, box = overlays[name]
    base = np.array(img_r).astype(float)

    # 초록 얼굴 윤곽, 빨강 헤어, 파랑 생성 대상
    for arr, ch in ((fm, 1), (hm, 0), (gm, 2)):
        m = np.array(arr) > 127
        base[m, ch] = base[m, ch] * 0.4 + 255 * 0.6

    ax.imshow(base.astype(np.uint8))
    x1, y1, x2, y2 = box
    ax.plot([x1, x2, x2, x1, x1], [y1, y1, y2, y2, y1], "y-", lw=1.5)
    ax.set_title(f"{name[6:16]}  L{LATENT[name]}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
savefig(85, "mask_overlay_upper")
plt.show()

## 기준 비율 산출

HAIR_DILATE_PX = 20 은 얼빡 세트(normal)로 정한 값이다. 그 세트의 1024 좌표계
얼굴 폭을 재서 20px 이 얼굴 폭의 몇 퍼센트였는지 구한다.

그 비율을 upper 세트에 적용하면 얼굴 크기와 무관하게 같은 효과가 난다.
InsightFace 박스 기준으로 재야 upper 측정과 비교가 된다.

In [ ]:
NORMAL = Path("/content/drive/MyDrive/saloncut_data/test_images/normal")

rows_n = []
for path in sorted(NORMAL.glob("*.jpg")):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    scale = 1024 / max(w, h)
    img_r = img.resize(((int(w * scale) // 8) * 8, (int(h * scale) // 8) * 8))

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    if not det:
        rows_n.append((path.stem, None, None))
        continue
    x1, _, x2, _ = det[0].bbox
    fw = x2 - x1
    rows_n.append((path.stem, fw, 20 / fw))

print(f"{'file':<26}{'얼굴폭':>9}{'20px 비율':>11}")
for name, fw, r in rows_n:
    if fw is None:
        print(f"{name:<26}{'검출실패':>9}")
        continue
    print(f"{name:<26}{fw:>9.0f}{r:>11.4f}")

vals = [r for _, fw, r in rows_n if fw]
print(f"\n비율   최소 {min(vals):.4f}   중앙 {sorted(vals)[len(vals)//2]:.4f}   최대 {max(vals):.4f}")
print(f"upper 세트 얼굴폭 102~226 에 적용하면 팽창 {min(vals)*102:.1f} ~ {max(vals)*226:.1f}px")

## 비율 팽창 검증

HAIR_DILATE_PX = 20 을 얼굴 폭 비례로 바꿔 다시 돌린다.

비율 0.077 은 normal 세트에서 앞머리가 있는 사진 3장 중 최대값이다.
20px 이 그 사진들에서 얼굴 폭의 몇 퍼센트였는지가 기준이고, 최대값을 택해
팽창을 덜 줄이는 보수적인 쪽으로 잡았다.

코사인이 내려가면 팽창이 원인이라는 것이 검증된다. 다만 팽창을 줄이면
잔머리 틈이 인페인팅 대상에 들어가 앞머리가 두꺼워질 수 있다. 그 부작용은
이미지로 따로 확인해야 한다.

In [ ]:
DILATE_RATIO = 0.077  # normal 앞머리 3장 중 최대

FIX_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/dilate_ratio")
FIX_DIR.mkdir(parents=True, exist_ok=True)

rows_fix = []
for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    dilate = max(1, int(fw * DILATE_RATIO))

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=dilate)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    b = compose.color_transfer(out, img_r, gen_mask)
    c = compose.align_then_recompose(img_r, b, face_mask, hair_mask)
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    d.save(FIX_DIR / f"c3_{path.stem}.png")
    rows_fix.append((path.stem, int(fw), dilate, identity(img_r, d)))
    print(f"{path.stem:<26} 얼굴폭 {int(fw):>3}  dilate {dilate:>2}px")

# --- 비교 ---

before = dict((n, v[3]) for n, v in rows)
print(f"\n{'file':<26}{'얼굴폭':>8}{'팽창':>7}{'기존20':>10}{'비율':>10}{'변화':>10}")
for name, fw, dl, aft in rows_fix:
    print(
        f"{name:<26}{fw:>8}{dl:>7}"
        f"{before[name]:>10.4f}{aft:>10.4f}{aft - before[name]:>+10.4f}"
    )

## 팽창 20px vs 비율 비교

코사인이 6장 모두 내려갔지만 upper_04·05 는 여전히 0.61·0.71 로 높다.
8/12 실험에서 MIXED_CLONE 을 0.71~0.84 로 "회피 실패" 판정한 전례가 있어
upper_05 는 아직 그 구간이다. 남은 원인이 있다.

단계별 코사인을 비율 팽창 조건에서 다시 재고, 얼굴을 확대해 실제로 무엇이
달라졌는지 본다. 앞머리가 두꺼워지는 부작용도 같이 확인한다.

In [ ]:
rows_stage2 = []
imgs2 = {}

for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    dilate = max(1, int(fw * DILATE_RATIO))

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=dilate)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    a = out
    b = compose.color_transfer(a, img_r, gen_mask)
    c = compose.align_then_recompose(img_r, b, face_mask, hair_mask)
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    imgs2[path.stem] = (img_r, d)
    rows_stage2.append((path.stem, [identity(img_r, x) for x in (a, b, c, d)]))

print(f"{'file':<26}" + "".join(f"{s:>12}" for s in STAGES) + f"{'재합성분':>10}{'고주파분':>10}")
for name, v in rows_stage2:
    print(
        f"{name:<26}" + "".join(f"{x:>12.4f}" for x in v)
        + f"{v[2] - v[1]:>+10.4f}{v[3] - v[2]:>+10.4f}"
    )

In [ ]:
def face_crop(img, pad=1.5, size=420):
    """얼굴 박스 주변을 잘라 확대한다."""
    det = gen_loader.get_face_app().get(np.array(img)[:, :, ::-1])
    if not det:
        return None
    x1, y1, x2, y2 = det[0].bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    half = max(x2 - x1, y2 - y1) / 2 * pad
    box = (
        max(0, int(cx - half)), max(0, int(cy - half)),
        min(img.width, int(cx + half)), min(img.height, int(cy + half)),
    )
    c = img.crop(box)
    return c.resize((size, int(size * c.height / c.width)), Image.LANCZOS)


LABELS = ("원본", "팽창 20px", "비율 팽창")
fig, axes = plt.subplots(3, 6, figsize=(22, 12))

for i, name in enumerate(order):
    img_r = imgs2[name][0]
    srcs = (
        img_r,
        Image.open(FINAL / f"c3_{name}.png").convert("RGB"),
        Image.open(FIX_DIR / f"c3_{name}.png").convert("RGB"),
    )
    for r, (lab, im) in enumerate(zip(LABELS, srcs)):
        c = face_crop(im)
        axes[r, i].imshow(c if c else Image.new("RGB", (420, 420)))
        axes[r, i].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, i].axis("off")

plt.tight_layout()
savefig(86, "dilate_ratio_face_compare")
plt.show()

In [ ]:
BANG = ["upper_02_west_small_face", "upper_04_asian_landscape", "upper_05_asian_glasses"]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i, name in enumerate(BANG):
    img_r = imgs2[name][0]
    srcs = (
        img_r,
        Image.open(FINAL / f"c3_{name}.png").convert("RGB"),
        Image.open(FIX_DIR / f"c3_{name}.png").convert("RGB"),
    )
    for r, (lab, im) in enumerate(zip(LABELS, srcs)):
        c = face_crop(im, pad=1.2, size=380)
        axes[r, i].imshow(c)
        axes[r, i].set_title(f"{lab}  {name[6:16]}", fontsize=9)
        axes[r, i].axis("off")

plt.tight_layout()
savefig(87, "dilate_ratio_bang_check")
plt.show()

## 색 얼룩 원인 추적

확대에서 콧대의 세로 밝은 띠와 뺨의 색 얼룩이 보인다. 원본에 없던 것이다.
코사인은 신원만 보므로 이 현상을 못 잡는다.

단계 사이의 ΔE 를 공간 지도로 그려 어느 단계가 어디를 바꾸는지 본다.
콧대 띠가 특정 단계의 지도에서만 나타나면 그 단계가 원인이다.

  A→B 색 정합
  B→C 어파인 정렬·재합성
  C→D 고주파 이식

고주파 이식은 원본에서 (원본 − 블러)를 뽑아 더한다. 생성된 코 모양이 원본과
다르면 원본의 콧대 하이라이트 경계가 어긋난 자리에 얹힐 수 있다.

In [ ]:
import cv2

STAGE2_DIR = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/stages_ratio")
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

stages_r = {}
for path in files:
    ref_id = REF_MAP.get(path.stem, "ref-01")
    out, img_r, _ = combo3.generate(
        Image.open(path).convert("RGB"), REF / f"{ref_id}.png", SEED
    )

    det = gen_loader.get_face_app().get(np.array(img_r)[:, :, ::-1])
    fw = det[0].bbox[2] - det[0].bbox[0]
    dilate = max(1, int(fw * DILATE_RATIO))

    face_mask = masks.build_face_mask(img_r)
    hair_mask = masks.build_hair_mask(img_r, dilate=dilate)
    gen_mask = masks.build_gen_mask(face_mask, hair_mask)

    a = out
    b = compose.color_transfer(a, img_r, gen_mask)
    c = compose.align_then_recompose(img_r, b, face_mask, hair_mask)
    d = compose.transfer_high_freq(c, img_r, gen_mask)

    for tag, im in zip(STAGES, (a, b, c, d)):
        im.save(STAGE2_DIR / f"{path.stem}_{tag}.png")

    stages_r[path.stem] = (img_r, a, b, c, d, gen_mask, det[0].bbox)
    print(f"{path.stem} 저장")

In [ ]:
def to_lab(img):
    return cv2.cvtColor(np.array(img).astype(np.float32) / 255, cv2.COLOR_RGB2LAB)


def delta_e(x, y):
    return np.linalg.norm(to_lab(x) - to_lab(y), axis=2)


def crop_box(bbox, img, pad=1.4):
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    h = max(x2 - x1, y2 - y1) / 2 * pad
    return (
        max(0, int(cx - h)), max(0, int(cy - h)),
        min(img.width, int(cx + h)), min(img.height, int(cy + h)),
    )


TRANS = (("A→B 색정합", 1, 2), ("B→C 재합성", 2, 3), ("C→D 고주파", 3, 4))

print(f"{'file':<26}" + "".join(f"{t[0]:>14}" for t in TRANS))
for name in order:
    v = stages_r[name]
    m = np.array(v[5]) > 127
    vals = [delta_e(v[i], v[j])[m].mean() for _, i, j in TRANS]
    print(f"{name:<26}" + "".join(f"{x:>14.2f}" for x in vals))

# --- 지도 ---

fig, axes = plt.subplots(4, 6, figsize=(22, 16))
for col, name in enumerate(order):
    v = stages_r[name]
    box = crop_box(v[6], v[0])

    axes[0, col].imshow(v[4].crop(box))
    axes[0, col].set_title(f"최종  {name[6:16]}", fontsize=9)
    axes[0, col].axis("off")

    for r, (lab, i, j) in enumerate(TRANS, start=1):
        de = delta_e(v[i], v[j])[box[1]:box[3], box[0]:box[2]]
        im = axes[r, col].imshow(de, cmap="inferno", vmin=0, vmax=25)
        axes[r, col].set_title(lab, fontsize=9)
        axes[r, col].axis("off")

plt.tight_layout()
savefig(88, "delta_e_stage_map")
plt.show()

## 어파인 정렬 이동량

재합성 단계의 ΔE 가 가장 크고 지도에서 얼굴 윤곽 전체가 밝다. 색이 망가진 것이
아니라 얼굴이 통째로 이동해서 나온 값일 수 있다. 둘은 지금 지표로 구분되지 않는다.

align_then_recompose() 의 어파인 행렬을 그대로 계산해 평행이동·회전·스케일을 뺀다.
이동량이 얼굴 폭 대비 몇 퍼센트인지 보면, HAIR_DILATE_PX·MIN_FACE_WIDTH 와 같은
"절대값이 작은 얼굴에서 과하게 작용" 구조인지 확인된다.

In [ ]:
from src.ai_engine.image_gen import compose as comp_mod

rows_af = []
for name in order:
    img_r, a, b, c, d, gen_mask, bbox = stages_r[name]

    o = np.array(img_r.convert("RGB"))
    g = np.array(b.resize(img_r.size).convert("RGB"))

    kps_o, kps_g = comp_mod._get_kps(o), comp_mod._get_kps(g)
    if kps_o is None or kps_g is None:
        rows_af.append((name, None, None, None, None))
        continue

    m, _ = cv2.estimateAffinePartial2D(kps_g, kps_o, method=cv2.LMEDS)
    if m is None:
        rows_af.append((name, None, None, None, None))
        continue

    tx, ty = m[0, 2], m[1, 2]
    scale = np.hypot(m[0, 0], m[1, 0])
    angle = np.degrees(np.arctan2(m[1, 0], m[0, 0]))
    fw = bbox[2] - bbox[0]

    rows_af.append((name, np.hypot(tx, ty), scale, angle, fw))

print(f"{'file':<26}{'이동px':>9}{'얼굴폭':>8}{'이동/폭':>9}{'스케일':>9}{'회전도':>9}")
for name, shift, sc, ang, fw in rows_af:
    if shift is None:
        print(f"{name:<26}{'정렬실패':>9}")
        continue
    print(f"{name:<26}{shift:>9.1f}{fw:>8.0f}{shift / fw:>9.3f}{sc:>9.3f}{ang:>9.2f}")

## 어파인 정렬 유무 비교

이동/폭 상위 2장(upper_06 0.256, upper_02 0.115)이 재합성 ΔE 상위 2장과 일치한다.
재합성 단계의 큰 ΔE 는 색이 망가진 것이 아니라 얼굴이 이동한 것이다.

정렬을 끄고 recompose_with_hair() 만 쓴 결과와 나란히 본다. 조합 5 가 쓰는
경로와 같다. 코사인·ΔE·얼굴 확대를 함께 보고 어느 쪽이 나은지 정한다.

정렬은 원래 img2img 로 얼굴이 미세하게 어긋나는 것을 보정하려는 것이므로,
끄면 그 어긋남이 남는다. 확대에서 이중 윤곽이 보이는지 확인해야 한다.

In [ ]:
NOALIGN = Path("/content/drive/MyDrive/saloncut_data/outputs/upper_combo3/no_align")
NOALIGN.mkdir(parents=True, exist_ok=True)

rows_na = []
noalign = {}

for name in order:
    img_r, a, b, c, d, gen_mask, bbox = stages_r[name]

    face_mask = masks.build_face_mask(img_r)
    fw = bbox[2] - bbox[0]
    hair_mask = masks.build_hair_mask(img_r, dilate=max(1, int(fw * DILATE_RATIO)))

    c2 = compose.recompose_with_hair(img_r, b.resize(img_r.size), face_mask, hair_mask)
    d2 = compose.transfer_high_freq(c2, img_r, gen_mask)

    d2.save(NOALIGN / f"c3_{name}.png")
    noalign[name] = (c2, d2)

    m = np.array(gen_mask) > 127
    rows_na.append(
        (
            name,
            identity(img_r, d),      # 정렬 있음
            identity(img_r, d2),     # 정렬 없음
            delta_e(b, c)[m].mean(),
            delta_e(b, c2)[m].mean(),
        )
    )

print(f"{'file':<26}{'코사인정렬':>12}{'코사인끔':>11}{'변화':>10}{'ΔE정렬':>10}{'ΔE끔':>9}")
for name, ci, cn, ei, en in rows_na:
    print(f"{name:<26}{ci:>12.4f}{cn:>11.4f}{cn - ci:>+10.4f}{ei:>10.2f}{en:>9.2f}")